In [22]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

path  = path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\india_equities_historical_final_updated_funds.parquet"
table = pq.read_table(path)

stocks_historic = table.to_pandas()

micro_indicators = pd.read_csv(
    r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\micro_indicators_monthly.csv"
)

print("Stocks:")
print(stocks_historic.head())



Stocks:
        Date      Open      High       Low      Close  Adj Close     Volume  \
0 2002-07-01  25.41818  25.81818  24.81818  25.127272  17.835777  2047540.0   
1 2002-07-01  25.41818  25.81818  24.81818  25.127272  17.835777  2047540.0   
2 2002-07-01  25.41818  25.81818  24.81818  25.127272  17.835777  2047540.0   
3 2002-07-01  25.41818  25.81818  24.81818  25.127272  17.835777  2047540.0   
4 2002-07-01  25.41818  25.81818  24.81818  25.127272  17.835777  2047540.0   

   Dividends  Stock Splits  yahoo_symbol  Capital Gains          isin  \
0        0.0           0.0  ICICIBANK.NS            NaN  INE090A01021   
1        0.0           0.0  ICICIBANK.NS            NaN  INE090A01021   
2        0.0           0.0  ICICIBANK.NS            NaN  INE090A01021   
3        0.0           0.0  ICICIBANK.NS            NaN  INE090A01021   
4        0.0           0.0  ICICIBANK.NS            NaN  INE090A01021   

      symbol series segment  
0  ICICIBANK     EQ  EQUITY  
1  ICICIBANK     E

In [23]:
print("\nMicro Indicators:")
print(micro_indicators.head())


Micro Indicators:
         date  oil_price_usd_bbl  gold_price_inr_10g  cpi_index  \
0  2010-01-01          71.459999        16050.981650        NaN   
1  2010-02-01          77.589996        16563.372599        NaN   
2  2010-03-01          82.699997        16035.455348        NaN   
3  2010-04-01          87.440002        16788.934644        NaN   
4  2010-05-01          74.019997        18067.165857        NaN   

   cpi_inflation_yoy_pct  house_price_index  oil_return_1m_pct  \
0                    NaN                NaN                NaN   
1                    NaN                NaN           8.578222   
2                    NaN                NaN           6.585901   
3                    NaN          90.626706           5.731567   
4                    NaN          90.626706         -15.347673   

   oil_return_12m_pct  gold_return_1m_pct  gold_return_12m_pct  \
0                 NaN                 NaN                  NaN   
1                 NaN            3.192272        

In [24]:
import pandas as pd
import numpy as np

# -----------------------------
# BASIC INFORMATION
# -----------------------------

print("=" * 60)
print("STOCK DATA")
print("=" * 60)

print("Shape:", stocks_historic.shape)
print("\nData types:")
print(stocks_historic.dtypes)

print("\nMissing values:")
print(stocks_historic.isna().sum())

print("\nDuplicate rows:", stocks_historic.duplicated().sum())

print("\nDate range:")
print("Min:", stocks_historic["Date"].min())
print("Max:", stocks_historic["Date"].max())

print("\nUnique stocks:")
print("ISIN:", stocks_historic["isin"].nunique())
print("Symbol:", stocks_historic["symbol"].nunique())


print("\n" + "=" * 60)
print("MICRO INDICATORS")
print("=" * 60)

print("Shape:", micro_indicators.shape)
print("\nData types:")
print(micro_indicators.dtypes)

print("\nMissing values:")
print(micro_indicators.isna().sum())

print("\nDuplicate rows:", micro_indicators.duplicated().sum())

print("\nDate range:")
print("Min:", micro_indicators["date"].min())
print("Max:", micro_indicators["date"].max())

STOCK DATA
Shape: (3915791, 15)

Data types:
Date             datetime64[ns]
Open                    float64
High                    float64
Low                     float64
Close                   float64
Adj Close               float64
Volume                  float64
Dividends               float64
Stock Splits            float64
yahoo_symbol                str
Capital Gains           float64
isin                        str
symbol                      str
series                      str
segment                     str
dtype: object

Missing values:
Date                   0
Open                   3
High                   3
Low                    3
Close                  3
Adj Close              3
Volume                 0
Dividends              0
Stock Splits           0
yahoo_symbol           0
Capital Gains    3910247
isin                   0
symbol                 0
series                 0
segment                0
dtype: int64

Duplicate rows: 2237988

Date range:
Min: 1991-01-02 00

In [25]:
# Exact duplicate rows
exact_duplicates = stocks_historic.duplicated(keep=False)

print("Exact duplicate rows:", exact_duplicates.sum())


Exact duplicate rows: 3017708


In [26]:
stocks_clean = stocks_historic.drop_duplicates().copy()

In [27]:
print("Original:", stocks_historic.shape)
print("Clean:", stocks_clean.shape)
print("Removed:", len(stocks_historic) - len(stocks_clean))

Original: (3915791, 15)
Clean: (1677803, 15)
Removed: 2237988


In [28]:
stock_date_count = (
    stocks_clean
    .groupby(["isin", "Date"])
    .size()
)

print("Maximum observations per ISIN-Date:",
      stock_date_count.max())

print(
    "ISIN-Date combinations with >1 observation:",
    (stock_date_count > 1).sum()
)

Maximum observations per ISIN-Date: 1
ISIN-Date combinations with >1 observation: 0


In [29]:
missing_ohlc = stocks_clean[
    stocks_clean[
        ["Open", "High", "Low", "Close", "Adj Close"]
    ].isna().any(axis=1)
]

print(missing_ohlc.to_string(index=False))

      Date  Open  High  Low  Close  Adj Close  Volume  Dividends  Stock Splits yahoo_symbol  Capital Gains         isin   symbol series segment
2026-09-17   NaN   NaN  NaN    NaN        NaN     0.0        4.0           0.0      TNPL.NS            NaN INE107A01015     TNPL     EQ  EQUITY
2026-09-17   NaN   NaN  NaN    NaN        NaN     0.0        4.0           0.0   SANSERA.NS            NaN INE953O01021  SANSERA     EQ  EQUITY
2026-09-17   NaN   NaN  NaN    NaN        NaN     0.0        1.5           0.0  LANDMARK.NS            NaN INE559R01029 LANDMARK     EQ  EQUITY


In [30]:
# 1. Remove exact duplicate records
stocks_clean = stocks_historic.drop_duplicates().copy()

# 2. Remove records without price information
stocks_clean = stocks_clean.dropna(
    subset=["Open", "High", "Low", "Close", "Adj Close"]
).copy()

# 3. Remove almost completely empty feature
stocks_clean = stocks_clean.drop(
    columns=["Capital Gains"]
)

# 4. Ensure correct date type
stocks_clean["Date"] = pd.to_datetime(
    stocks_clean["Date"],
    errors="coerce"
)

# 5. Sort
stocks_clean = (
    stocks_clean
    .sort_values(["isin", "Date"])
    .reset_index(drop=True)
)

In [31]:
print("Final shape:", stocks_clean.shape)

print(
    "Exact duplicates:",
    stocks_clean.duplicated().sum()
)

stock_date_count = (
    stocks_clean
    .groupby(["isin", "Date"])
    .size()
)

print(
    "Maximum records per ISIN-Date:",
    stock_date_count.max()
)

print(
    "ISIN-Date combinations with >1 record:",
    (stock_date_count > 1).sum()
)

Final shape: (1677800, 14)
Exact duplicates: 0
Maximum records per ISIN-Date: 1
ISIN-Date combinations with >1 record: 0


In [32]:
import pyarrow as pa
import pyarrow.parquet as pq

output_path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\stocks_historical_clean.parquet"

table = pa.Table.from_pandas(
    stocks_clean,
    preserve_index=False
)

pq.write_table(
    table,
    output_path
)

print("Saved successfully")

Saved successfully


In [33]:
pf = pq.ParquetFile(output_path)

print("Rows:", pf.metadata.num_rows)
print("Columns:", pf.metadata.num_columns)

Rows: 1677800
Columns: 14


## micro cleaning

In [34]:
micro_clean = micro_indicators.copy()

micro_clean["date"] = pd.to_datetime(
    micro_clean["date"],
    errors="coerce"
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

print(micro_clean.dtypes)
print("\nDate range:")
print(micro_clean["date"].min())
print(micro_clean["date"].max())

print("\nDuplicate dates:")
print(micro_clean["date"].duplicated().sum())

date                     datetime64[us]
oil_price_usd_bbl               float64
gold_price_inr_10g              float64
cpi_index                       float64
cpi_inflation_yoy_pct           float64
house_price_index               float64
oil_return_1m_pct               float64
oil_return_12m_pct              float64
gold_return_1m_pct              float64
gold_return_12m_pct             float64
inflation_change_1m             float64
hpi_growth_12m_pct              float64
dtype: object

Date range:
2010-01-01 00:00:00
2026-09-01 00:00:00

Duplicate dates:
0


In [35]:
for col in micro_clean.columns:
    if col == "date":
        continue

    missing_dates = micro_clean.loc[
        micro_clean[col].isna(),
        "date"
    ]

    print(f"\n{'=' * 60}")
    print(col)
    print("Missing:", len(missing_dates))

    if len(missing_dates) > 0:
        print(
            missing_dates.dt.strftime("%Y-%m").tolist()
        )


oil_price_usd_bbl
Missing: 29
['2010-08', '2011-05', '2012-01', '2012-04', '2012-07', '2013-09', '2013-12', '2014-06', '2015-02', '2015-03', '2015-11', '2016-05', '2017-01', '2017-10', '2018-04', '2018-07', '2019-09', '2019-12', '2020-03', '2020-11', '2021-08', '2022-05', '2023-01', '2023-10', '2024-09', '2024-12', '2025-06', '2026-02', '2026-03']

gold_price_inr_10g
Missing: 29
['2010-08', '2011-05', '2012-01', '2012-04', '2012-07', '2013-09', '2013-12', '2014-06', '2015-02', '2015-03', '2015-11', '2016-05', '2017-01', '2017-10', '2018-04', '2018-07', '2019-09', '2019-12', '2020-03', '2020-11', '2021-08', '2022-05', '2023-01', '2023-10', '2024-09', '2024-12', '2025-06', '2026-02', '2026-03']

cpi_index
Missing: 45
['2010-01', '2010-02', '2010-03', '2010-04', '2010-05', '2010-06', '2010-07', '2010-08', '2010-09', '2010-10', '2010-11', '2010-12', '2011-01', '2011-02', '2011-03', '2011-04', '2011-05', '2011-06', '2011-07', '2011-08', '2011-09', '2011-10', '2011-11', '2011-12', '2012-01'

In [36]:
summary = pd.DataFrame({
    "dtype": micro_clean.dtypes,
    "missing": micro_clean.isna().sum(),
    "missing_pct": (
        micro_clean.isna().mean() * 100
    ).round(2),
    "valid": micro_clean.notna().sum()
})

print(summary)

                                dtype  missing  missing_pct  valid
date                   datetime64[us]        0         0.00    201
oil_price_usd_bbl             float64       29        14.43    172
gold_price_inr_10g            float64       29        14.43    172
cpi_index                     float64       45        22.39    156
cpi_inflation_yoy_pct         float64       59        29.35    142
house_price_index             float64        3         1.49    198
oil_return_1m_pct             float64        1         0.50    200
oil_return_12m_pct            float64       12         5.97    189
gold_return_1m_pct            float64        1         0.50    200
gold_return_12m_pct           float64       12         5.97    189
inflation_change_1m           float64       61        30.35    140
hpi_growth_12m_pct            float64       15         7.46    186


In [37]:
micro_clean = micro_indicators.copy()

micro_clean["date"] = pd.to_datetime(
    micro_clean["date"]
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

In [38]:
print(
    micro_clean[
        micro_clean["date"].dt.strftime("%Y-%m").isin([
            "2010-01",
            "2010-02",
            "2010-03",
            "2010-04",
            "2010-05",
            "2011-03",
            "2011-04"
        ])
    ][
        [
            "date",
            "house_price_index",
            "hpi_growth_12m_pct"
        ]
    ].to_string(index=False)
)

      date  house_price_index  hpi_growth_12m_pct
2010-01-01                NaN                 NaN
2010-02-01                NaN                 NaN
2010-03-01                NaN                 NaN
2010-04-01          90.626706                 NaN
2010-05-01          90.626706                 NaN
2011-03-01         108.776360                 NaN
2011-04-01         122.091899           34.719559


In [39]:
micro_clean = micro_indicators.copy()

micro_clean["date"] = pd.to_datetime(
    micro_clean["date"]
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

In [40]:
micro_clean["oil_price_usd_bbl"] = (
    micro_clean["oil_price_usd_bbl"]
    .interpolate(method="linear")
)

micro_clean["gold_price_inr_10g"] = (
    micro_clean["gold_price_inr_10g"]
    .interpolate(method="linear")
)

In [41]:
print(
    micro_clean[
        micro_clean["date"].dt.strftime("%Y-%m").isin(
            ["2010-07", "2010-08", "2010-09",
             "2011-04", "2011-05", "2011-06"]
        )
    ][
        ["date", "oil_price_usd_bbl", "gold_price_inr_10g"]
    ].to_string(index=False)
)

      date  oil_price_usd_bbl  gold_price_inr_10g
2010-07-01          78.180000        17613.339759
2010-08-01          80.244999        18202.015416
2010-09-01          82.309998        18790.691074
2011-04-01         125.889999        22161.766434
2011-05-01         119.185001        21890.438061
2011-06-01         112.480003        21619.109689


In [42]:
micro_clean["oil_return_1m_pct"] = (
    micro_clean["oil_price_usd_bbl"]
    .pct_change() * 100
)

micro_clean["oil_return_12m_pct"] = (
    micro_clean["oil_price_usd_bbl"]
    .pct_change(periods=12) * 100
)

micro_clean["gold_return_1m_pct"] = (
    micro_clean["gold_price_inr_10g"]
    .pct_change() * 100
)

micro_clean["gold_return_12m_pct"] = (
    micro_clean["gold_price_inr_10g"]
    .pct_change(periods=12) * 100
)

In [43]:
print(
    micro_clean.head(15)[
        [
            "date",
            "oil_price_usd_bbl",
            "oil_return_1m_pct",
            "oil_return_12m_pct",
            "gold_price_inr_10g",
            "gold_return_1m_pct",
            "gold_return_12m_pct"
        ]
    ].to_string(index=False)
)

      date  oil_price_usd_bbl  oil_return_1m_pct  oil_return_12m_pct  gold_price_inr_10g  gold_return_1m_pct  gold_return_12m_pct
2010-01-01          71.459999                NaN                 NaN        16050.981650                 NaN                  NaN
2010-02-01          77.589996           8.578222                 NaN        16563.372599            3.192272                  NaN
2010-03-01          82.699997           6.585901                 NaN        16035.455348           -3.187257                  NaN
2010-04-01          87.440002           5.731567                 NaN        16788.934644            4.698833                  NaN
2010-05-01          74.019997         -15.347673                 NaN        18067.165857            7.613534                  NaN
2010-06-01          75.010002           1.337484                 NaN        18580.302862            2.840163                  NaN
2010-07-01          78.180000           4.226101                 NaN        17613.339759  

In [44]:
for col in ["oil_price_usd_bbl", "gold_price_inr_10g"]:
    missing = micro_clean[col].isna()

    groups = (
        missing.ne(missing.shift())
        .cumsum()
    )

    gap_lengths = (
        missing[missing]
        .groupby(groups[missing])
        .size()
    )

    print(
        col,
        "maximum consecutive missing months:",
        gap_lengths.max()
    )

oil_price_usd_bbl maximum consecutive missing months: nan
gold_price_inr_10g maximum consecutive missing months: nan


In [45]:
print("Oil missing:",
      micro_clean["oil_price_usd_bbl"].isna().sum())

print("Gold missing:",
      micro_clean["gold_price_inr_10g"].isna().sum())

print("Oil 1M return missing:",
      micro_clean["oil_return_1m_pct"].isna().sum())

print("Oil 12M return missing:",
      micro_clean["oil_return_12m_pct"].isna().sum())

print("Gold 1M return missing:",
      micro_clean["gold_return_1m_pct"].isna().sum())

print("Gold 12M return missing:",
      micro_clean["gold_return_12m_pct"].isna().sum())

Oil missing: 0
Gold missing: 0
Oil 1M return missing: 1
Oil 12M return missing: 12
Gold 1M return missing: 1
Gold 12M return missing: 12


In [46]:
print(
    micro_clean[
        [
            "date",
            "cpi_index",
            "cpi_inflation_yoy_pct",
            "inflation_change_1m"
        ]
    ].to_string(index=False)
)

      date  cpi_index  cpi_inflation_yoy_pct  inflation_change_1m
2010-01-01        NaN                    NaN                  NaN
2010-02-01        NaN                    NaN                  NaN
2010-03-01        NaN                    NaN                  NaN
2010-04-01        NaN                    NaN                  NaN
2010-05-01        NaN                    NaN                  NaN
2010-06-01        NaN                    NaN                  NaN
2010-07-01        NaN                    NaN                  NaN
2010-08-01        NaN                    NaN                  NaN
2010-09-01        NaN                    NaN                  NaN
2010-10-01        NaN                    NaN                  NaN
2010-11-01        NaN                    NaN                  NaN
2010-12-01        NaN                    NaN                  NaN
2011-01-01        NaN                    NaN                  NaN
2011-02-01        NaN                    NaN                  NaN
2011-03-01

In [47]:
check = [
    "2013-12",
    "2014-01",
    "2014-02",
    "2020-03",
    "2020-04",
    "2020-05",
    "2020-06",
    "2025-12",
    "2026-01",
    "2026-02",
    "2026-09"
]

print(
    micro_clean[
        micro_clean["date"].dt.strftime("%Y-%m").isin(check)
    ][
        [
            "date",
            "cpi_index",
            "cpi_inflation_yoy_pct",
            "inflation_change_1m"
        ]
    ].to_string(index=False)
)

      date  cpi_index  cpi_inflation_yoy_pct  inflation_change_1m
2013-12-01      114.5                    NaN                  NaN
2014-01-01      113.6                   8.60                  NaN
2014-02-01      113.6                   7.88                -0.72
2020-03-01      148.6                   5.84                -0.74
2020-04-01      151.4                    NaN                  NaN
2020-05-01      150.9                    NaN                  NaN
2020-06-01      151.8                   6.23                  NaN
2025-12-01      198.0                   1.33                 0.62
2026-01-01        NaN                    NaN                  NaN
2026-02-01        NaN                    NaN                  NaN
2026-09-01        NaN                    NaN                  NaN


In [48]:
# Calculate CPI YoY inflation from CPI index
cpi_calculated_inflation = (
    micro_clean["cpi_index"].pct_change(periods=12) * 100
)

# Only fill missing inflation values using the calculated value
micro_clean["cpi_inflation_yoy_pct"] = (
    micro_clean["cpi_inflation_yoy_pct"]
    .fillna(cpi_calculated_inflation)
)

In [49]:
print(
    micro_clean[
        micro_clean["date"].between("2020-03-01", "2020-06-01")
    ][
        ["date", "cpi_index", "cpi_inflation_yoy_pct"]
    ]
)

          date  cpi_index  cpi_inflation_yoy_pct
122 2020-03-01      148.6               5.840000
123 2020-04-01      151.4               7.223796
124 2020-05-01      150.9               6.267606
125 2020-06-01      151.8               6.230000


In [50]:
micro_clean["inflation_change_1m"] = (
    micro_clean["cpi_inflation_yoy_pct"].diff()
)
print(
    micro_clean[
        ["date", "cpi_inflation_yoy_pct", "inflation_change_1m"]
    ].tail(20)
)

          date  cpi_inflation_yoy_pct  inflation_change_1m
181 2025-02-01                   3.61                -0.65
182 2025-03-01                   3.34                -0.27
183 2025-04-01                   3.16                -0.18
184 2025-05-01                   2.82                -0.34
185 2025-06-01                   2.10                -0.72
186 2025-07-01                   1.61                -0.49
187 2025-08-01                   2.07                 0.46
188 2025-09-01                   1.44                -0.63
189 2025-10-01                   0.25                -1.19
190 2025-11-01                   0.71                 0.46
191 2025-12-01                   1.33                 0.62
192 2026-01-01                    NaN                  NaN
193 2026-02-01                    NaN                  NaN
194 2026-03-01                    NaN                  NaN
195 2026-04-01                    NaN                  NaN
196 2026-05-01                    NaN                  N

In [51]:
from openpyxl import load_workbook
import pandas as pd

path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\raw\cpi_887.xlsx"

wb = load_workbook(path, read_only=True, data_only=True)
ws = wb["CPI Data"]

# Read rows
rows = list(ws.values)

wb.close()

# First row = column names
headers = rows[0]

cpi_new = pd.DataFrame(rows[1:], columns=headers)

print("Shape:", cpi_new.shape)
print("Columns:")
print(cpi_new.columns.tolist())

print("\nFirst 5 rows:")
print(cpi_new.head().to_string(index=False))

Shape: (2200, 15)
Columns:
['base_year', 'series', 'year', 'month', 'state', 'sector', 'division', 'group', 'class', 'sub_class', 'item', 'code', 'index', 'inflation', 'imputation']

First 5 rows:
base_year  series year  month                       state   sector      division group class sub_class item code  index inflation imputation
     2024 Current 2026 August                   All India    Rural CPI (General)  None  None      None None None 109.27      5.23       None
     2024 Current 2026 August                   All India    Urban CPI (General)  None  None      None None None 108.07      4.31       None
     2024 Current 2026 August                   All India Combined CPI (General)  None  None      None None None 108.74      4.82       None
     2024 Current 2026 August Andaman And Nicobar Islands    Rural CPI (General)  None  None      None None None 110.57      4.09       None
     2024 Current 2026 August Andaman And Nicobar Islands    Urban CPI (General)  None  None      

In [52]:
cpi_2024 = cpi_new[
    (cpi_new["base_year"] == 2024) &
    (cpi_new["series"] == "Current") &
    (cpi_new["state"] == "All India") &
    (cpi_new["sector"] == "Combined") &
    (cpi_new["division"] == "CPI (General)")
].copy()

print("Rows:", len(cpi_2024))

print(
    cpi_2024[
        ["year", "month", "index", "inflation"]
    ].to_string(index=False)
)

Rows: 0
Empty DataFrame
Columns: [year, month, index, inflation]
Index: []


In [53]:
cpi_2024["date"] = pd.to_datetime(
    cpi_2024["year"].astype(str) + "-" +
    cpi_2024["month"].astype(str),
    format="%Y-%B"
)

cpi_2024 = cpi_2024.sort_values("date").reset_index(drop=True)

print(
    cpi_2024[
        ["date", "index", "inflation"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [date, index, inflation]
Index: []


In [54]:
print("Start:", cpi_2024["date"].min())
print("End:", cpi_2024["date"].max())

Start: NaT
End: NaT


In [55]:
cpi_2024 = cpi_new[
    (cpi_new["state"].astype(str).str.strip() == "All India") &
    (cpi_new["sector"].astype(str).str.strip() == "Combined")
].copy()

print(cpi_2024[[
    "year", "month", "index", "inflation"
]].to_string(index=False))

year     month  index inflation
2026    August 108.74      4.82
2026      July 107.95      4.45
2026      June 107.00      4.38
2026       May 105.91      3.93
2026     April 105.12      3.48
2026     March 104.84      3.40
2026  February 104.57      3.21
2026   January 104.45      2.74
2025  December 104.10       NaN
2025  November 104.01       NaN
2025   October 103.74       NaN
2025 September 103.74       NaN
2025    August 103.74       NaN
2025      July 103.35       NaN
2025      June 102.51       NaN
2025       May 101.90       NaN
2025     April 101.58       NaN
2025     March 101.39       NaN
2025  February 101.32       NaN
2025   January 101.67       NaN


In [56]:
cpi_2026 = cpi_new[
    (cpi_new["state"].astype(str).str.strip() == "All India") &
    (cpi_new["sector"].astype(str).str.strip() == "Combined")
].copy()

# Create date
cpi_2026["date"] = pd.to_datetime(
    cpi_2026["year"].astype(str) + "-" +
    cpi_2026["month"].astype(str),
    format="%Y-%B",
    errors="coerce"
)

# Keep required columns
cpi_2026 = cpi_2026[
    ["date", "index", "inflation"]
].sort_values("date").reset_index(drop=True)

print(cpi_2026.to_string(index=False))

      date  index inflation
2025-01-01 101.67       NaN
2025-02-01 101.32       NaN
2025-03-01 101.39       NaN
2025-04-01 101.58       NaN
2025-05-01 101.90       NaN
2025-06-01 102.51       NaN
2025-07-01 103.35       NaN
2025-08-01 103.74       NaN
2025-09-01 103.74       NaN
2025-10-01 103.74       NaN
2025-11-01 104.01       NaN
2025-12-01 104.10       NaN
2026-01-01 104.45      2.74
2026-02-01 104.57      3.21
2026-03-01 104.84      3.40
2026-04-01 105.12      3.48
2026-05-01 105.91      3.93
2026-06-01 107.00      4.38
2026-07-01 107.95      4.45
2026-08-01 108.74      4.82


In [57]:
print(
    cpi_2026[
        cpi_2026["date"].dt.year == 2026
    ].to_string(index=False)
)

      date  index inflation
2026-01-01 104.45      2.74
2026-02-01 104.57      3.21
2026-03-01 104.84      3.40
2026-04-01 105.12      3.48
2026-05-01 105.91      3.93
2026-06-01 107.00      4.38
2026-07-01 107.95      4.45
2026-08-01 108.74      4.82


In [58]:
micro_clean["inflation_change_1m"] = (
    micro_clean["cpi_inflation_yoy_pct"].diff()
)

In [59]:
print(
    micro_clean[
        micro_clean["date"].between("2025-12-01", "2026-09-01")
    ][
        ["date", "cpi_inflation_yoy_pct", "inflation_change_1m"]
    ].to_string(index=False)
)

      date  cpi_inflation_yoy_pct  inflation_change_1m
2025-12-01                   1.33                 0.62
2026-01-01                    NaN                  NaN
2026-02-01                    NaN                  NaN
2026-03-01                    NaN                  NaN
2026-04-01                    NaN                  NaN
2026-05-01                    NaN                  NaN
2026-06-01                    NaN                  NaN
2026-07-01                    NaN                  NaN
2026-08-01                    NaN                  NaN
2026-09-01                    NaN                  NaN


In [60]:
print("micro_clean:", micro_clean["date"].dtype)
print("cpi_2026:", cpi_2026["date"].dtype)

print("\nCPI dates:")
print(cpi_2026["date"].head().to_list())

print("\nMacro dates:")
print(
    micro_clean[
        micro_clean["date"].between("2026-01-01", "2026-08-01")
    ]["date"].to_list()
)

micro_clean: datetime64[us]
cpi_2026: datetime64[us]

CPI dates:
[Timestamp('2025-01-01 00:00:00'), Timestamp('2025-02-01 00:00:00'), Timestamp('2025-03-01 00:00:00'), Timestamp('2025-04-01 00:00:00'), Timestamp('2025-05-01 00:00:00')]

Macro dates:
[Timestamp('2026-01-01 00:00:00'), Timestamp('2026-02-01 00:00:00'), Timestamp('2026-03-01 00:00:00'), Timestamp('2026-04-01 00:00:00'), Timestamp('2026-05-01 00:00:00'), Timestamp('2026-06-01 00:00:00'), Timestamp('2026-07-01 00:00:00'), Timestamp('2026-08-01 00:00:00')]


In [61]:
print(
    cpi_2026[
        cpi_2026["date"].dt.year == 2026
    ][["date", "index", "inflation"]].to_string(index=False)
)

      date  index inflation
2026-01-01 104.45      2.74
2026-02-01 104.57      3.21
2026-03-01 104.84      3.40
2026-04-01 105.12      3.48
2026-05-01 105.91      3.93
2026-06-01 107.00      4.38
2026-07-01 107.95      4.45
2026-08-01 108.74      4.82


In [62]:
cpi_2026_only = cpi_2026[
    (cpi_2026["date"].dt.year == 2026) &
    (cpi_2026["date"].dt.month <= 8)
][
    ["date", "inflation"]
].copy()

cpi_2026_only = cpi_2026_only.rename(
    columns={"inflation": "cpi_inflation_new"}
)

print(cpi_2026_only.to_string(index=False))

      date cpi_inflation_new
2026-01-01              2.74
2026-02-01              3.21
2026-03-01              3.40
2026-04-01              3.48
2026-05-01              3.93
2026-06-01              4.38
2026-07-01              4.45
2026-08-01              4.82


In [63]:
micro_clean = micro_clean.merge(
    cpi_2026_only,
    on="date",
    how="left"
)

micro_clean["cpi_inflation_yoy_pct"] = (
    micro_clean["cpi_inflation_yoy_pct"]
    .fillna(micro_clean["cpi_inflation_new"])
)

micro_clean = micro_clean.drop(
    columns=["cpi_inflation_new"]
)

In [66]:
micro_clean["cpi_inflation_yoy_pct"] = pd.to_numeric(
    micro_clean["cpi_inflation_yoy_pct"],
    errors="coerce"
)
print(micro_clean["cpi_inflation_yoy_pct"].dtype)

float64


In [67]:
micro_clean["inflation_change_1m"] = (
    micro_clean["cpi_inflation_yoy_pct"].diff()
)

In [68]:
print(
    micro_clean[
        micro_clean["date"].between(
            "2025-12-01",
            "2026-09-01"
        )
    ][
        ["date", "cpi_inflation_yoy_pct", "inflation_change_1m"]
    ].to_string(index=False)
)

      date  cpi_inflation_yoy_pct  inflation_change_1m
2025-12-01                   1.33                 0.62
2026-01-01                   2.74                 1.41
2026-02-01                   3.21                 0.47
2026-03-01                   3.40                 0.19
2026-04-01                   3.48                 0.08
2026-05-01                   3.93                 0.45
2026-06-01                   4.38                 0.45
2026-07-01                   4.45                 0.07
2026-08-01                   4.82                 0.37
2026-09-01                    NaN                  NaN


In [70]:
micro_clean["hpi_growth_12m_pct"] = (
    micro_clean["house_price_index"].pct_change(periods=12) * 100
)
print(
    micro_clean[
        ["date", "house_price_index", "hpi_growth_12m_pct"]
    ].head(20).to_string(index=False)
)
print(
    micro_clean[
        ["date", "house_price_index", "hpi_growth_12m_pct"]
    ].tail(20).to_string(index=False)
)

      date  house_price_index  hpi_growth_12m_pct
2010-01-01                NaN                 NaN
2010-02-01                NaN                 NaN
2010-03-01                NaN                 NaN
2010-04-01          90.626706                 NaN
2010-05-01          90.626706                 NaN
2010-06-01          90.626706                 NaN
2010-07-01          99.717032                 NaN
2010-08-01          99.717032                 NaN
2010-09-01          99.717032                 NaN
2010-10-01         100.879902                 NaN
2010-11-01         100.879902                 NaN
2010-12-01         100.879902                 NaN
2011-01-01         108.776360                 NaN
2011-02-01         108.776360                 NaN
2011-03-01         108.776360                 NaN
2011-04-01         122.091899           34.719559
2011-05-01         122.091899           34.719559
2011-06-01         122.091899           34.719559
2011-07-01         131.417743           31.790669


In [72]:
print(
    micro_clean[
        micro_clean["date"].between("2024-01-01", "2025-06-01")
    ][
        ["date", "house_price_index", "hpi_growth_12m_pct"]
    ].to_string(index=False)
)
print(
    micro_clean[
        ["date", "house_price_index"]
    ].dropna().tail(40).to_string(index=False)
)
hpi_changes = (
    micro_clean["house_price_index"]
    .dropna()
    .ne(
        micro_clean["house_price_index"]
        .dropna()
        .shift()
    )
)

print("HPI observations:", micro_clean["house_price_index"].notna().sum())
print("HPI value changes:", hpi_changes.sum())

      date  house_price_index  hpi_growth_12m_pct
2024-01-01         294.181749            0.299062
2024-02-01         294.181749            0.299062
2024-03-01         294.181749            0.299062
2024-04-01         307.270150            0.270159
2024-05-01         307.270150            0.270159
2024-06-01         307.270150            0.270159
2024-07-01         303.617091            2.948661
2024-08-01         303.617091            2.948661
2024-09-01         303.617091            2.948661
2024-10-01         107.254140            5.692751
2024-11-01         107.254140            5.692751
2024-12-01         107.254140            5.692751
2025-01-01         101.252339          -65.581706
2025-02-01         101.252339          -65.581706
2025-03-01         101.252339          -65.581706
2025-04-01          99.341622          -67.669615
2025-05-01          99.341622          -67.669615
2025-06-01          99.341622          -67.669615
      date  house_price_index
2023-06-01         3

In [76]:
print(micro_clean[
    micro_clean["date"].between("2023-01-01", "2026-09-01")
][
    ["date", "house_price_index"]
].to_string(index=False))

      date  house_price_index
2023-01-01         293.304588
2023-02-01         293.304588
2023-03-01         293.304588
2023-04-01         306.442268
2023-05-01         306.442268
2023-06-01         306.442268
2023-07-01         294.920874
2023-08-01         294.920874
2023-09-01         294.920874
2023-10-01         101.477291
2023-11-01         101.477291
2023-12-01         101.477291
2024-01-01         294.181749
2024-02-01         294.181749
2024-03-01         294.181749
2024-04-01         307.270150
2024-05-01         307.270150
2024-06-01         307.270150
2024-07-01         303.617091
2024-08-01         303.617091
2024-09-01         303.617091
2024-10-01         107.254140
2024-11-01         107.254140
2024-12-01         107.254140
2025-01-01         101.252339
2025-02-01         101.252339
2025-03-01         101.252339
2025-04-01          99.341622
2025-05-01          99.341622
2025-06-01          99.341622
2025-07-01         111.312022
2025-08-01         111.312022
2025-09-01

In [77]:
import openpyxl
import pandas as pd

hpi_path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\raw\House Price Index Publication(1).xlsx"

wb = openpyxl.load_workbook(
    hpi_path,
    read_only=True,
    data_only=True
)

ws = wb["House Price Index"]

rows = list(ws.iter_rows(values_only=True))

# Find the section containing:
# "Base Year : 2022-23 = 100#"

new_base_row = None

for i, row in enumerate(rows):
    if row[1] is not None and "2022-23 = 100" in str(row[1]):
        new_base_row = i
        break

print("New HPI section starts at row:", new_base_row)

New HPI section starts at row: 90


c:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\backend\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [78]:
# Inspect rows around the new HPI section

for i in range(90, 110):
    print(i, rows[i])

90 (None, 'Base Year : 2022-23 = 100#', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
91 (None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
92 (None, None, 'Ahmedabad', 'Bangalore', 'Chandigarh', 'Chennai', 'Delhi', 'Gautam Buddha Nagar', 'Ghaziabad', 'Hyderabad', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai', 'Nagpur', 'Pune', 'Thane', 'Thiruvananthapuram')
93 (None, 'Q1.2022-23', 96.735422994312, 98.231695307756, 100.437870763503, 98.774299190852, 95.820013228749, 100.594993269091, 94.613703136123, 98.122229276989, 99.862488458546, 94.238260413514, 98.969405016268, 95.049431508333, 111.802831276138, 98.123985137719, 98.084586193775, 97.712134874075, 98.849909395213, 99.520122245004)
94 (None, 'Q2.2022-23', 105.425615300447, 97.369066727865, 101.976117693975, 101.556072558187, 101.979920013514, 101.384160772078, 99.021276728488, 100.42942612

In [79]:
for i in range(85, 93):
    print(i, rows[i])
for i, row in enumerate(rows):
    for j, value in enumerate(row):
        if value is not None and "All India" in str(value):
            print("row:", i, "col:", j, "value:", value)

85 (None, 'Q2.2024-25', 342.447026828968, 360.108892207036, 312.131105875744, 350.401842252501, 174.329550397329, 181.203911917184, 335.652421668499, 342.633446415831, 418.992287607405, 303.617090678304, 321.999921426744, None, None, None, None, None, None, None)
86 (None, 'Q3.2024-25', 351.161298442789, 362.441387983541, 315.239897710101, 346.197014298257, 177.426294497562, 180.015262944255, 337.839020336605, 345.062495364466, 420.935047347707, 304.401977501236, 323.255378988882, None, None, None, None, None, None, None)
87 (None, 'Q4.2024-25(P)**', 351.437790210445, 375.51404106394, 320.749220930645, 348.099473179416, 183.627753325021, 179.068541311583, 331.142080583108, 355.049129249616, 417.802765153853, 301.118147300734, 326.229666324121, None, None, None, None, None, None, None)
88 (None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
89 (None, None, None, None, None, None, None, None, None, None, None, None, None

In [81]:
for i in range(109, len(rows)):
    if any(x is not None for x in rows[i]):
        print(i, rows[i])
for i in range(0, 35):
    print(i, rows[i])

109 (None, 'Q1.2026-27(P)**', 118.833839406673, 127.076762446566, 147.261395530565, 166.847966670591, 96.222012538993, 103.077567736062, 122.588025207893, 103.521711863279, 145.408128405971, 126.618628296143, 119.63960464038, 108.072922152287, 137.084716316156, 108.243555878241, 114.004606717472, 112.442389295339, 114.37865205907, 122.170340029275)
111 (None, 'Note: #The base year of HPI has changed to 2022-23 from 2010-11. The coverage has increased from 10 cities to 18 cities viz., Mumbai, Delhi, Chennai, Kolkata, Bangalore, Lucknow, Ahmedabad, Jaipur, Kanpur, Kochi, Hyderabad, Thiruvananthapuram, Pune, Ghaziabad, Thane, Gautam Buddha Nagar, Chandigarh and Nagpur.\nAll India index is a weighted average of city indices, weights based on population proportion.\n**(P) implies Provisional.       \n', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
0 (None, None, None, None, None, None, None, None, None, None, None, None, None, N

In [82]:
hpi_rows = []

for row in rows[28:]:
    quarter = row[1]

    if quarter is None:
        continue

    quarter = str(quarter).strip()

    if not quarter.startswith("Q"):
        continue

    all_india = row[12]

    if all_india is None:
        continue

    hpi_rows.append({
        "quarter": quarter,
        "house_price_index": float(all_india)
    })

hpi_quarterly = pd.DataFrame(hpi_rows)

print(hpi_quarterly.tail(20).to_string(index=False))

        quarter  house_price_index
     Q2.2024-25         321.999921
     Q3.2024-25         323.255379
Q4.2024-25(P)**         326.229666
     Q1.2022-23          98.969405
     Q2.2022-23          97.501770
     Q3.2022-23         100.770697
     Q4.2022-23         102.758128
     Q1.2023-24          96.697261
     Q2.2023-24         102.247108
     Q3.2023-24         110.307286
     Q4.2023-24         114.402271
     Q1.2024-25         114.369276
     Q2.2024-25         109.967298
     Q3.2024-25         113.104928
     Q4.2024-25         105.818824
     Q1.2025-26         110.571820
     Q2.2025-26         118.732611
     Q3.2025-26         121.352007
     Q4.2025-26         117.830261
Q1.2026-27(P)**         119.639605


In [83]:
hpi_quarterly = hpi_quarterly.copy()

# Remove provisional marker
hpi_quarterly["quarter"] = (
    hpi_quarterly["quarter"]
    .str.replace("(P)**", "", regex=False)
    .str.strip()
)

print(hpi_quarterly.tail(10).to_string(index=False))

   quarter  house_price_index
Q4.2023-24         114.402271
Q1.2024-25         114.369276
Q2.2024-25         109.967298
Q3.2024-25         113.104928
Q4.2024-25         105.818824
Q1.2025-26         110.571820
Q2.2025-26         118.732611
Q3.2025-26         121.352007
Q4.2025-26         117.830261
Q1.2026-27         119.639605


In [84]:
quarter_months = {
    "Q1": [1, 2, 3],
    "Q2": [4, 5, 6],
    "Q3": [7, 8, 9],
    "Q4": [10, 11, 12]
}

monthly_hpi = []

for _, row in hpi_quarterly.iterrows():

    quarter, financial_year = row["quarter"].split(".")

    start_year = int(financial_year.split("-")[0])

    for month in quarter_months[quarter]:

        monthly_hpi.append({
            "date": pd.Timestamp(start_year, month, 1),
            "house_price_index": row["house_price_index"]
        })

monthly_hpi = pd.DataFrame(monthly_hpi)

monthly_hpi = monthly_hpi.sort_values("date").reset_index(drop=True)

print(monthly_hpi.tail(20).to_string(index=False))

      date  house_price_index
2024-10-01         105.818824
2024-11-01         326.229666
2024-11-01         105.818824
2024-12-01         326.229666
2024-12-01         105.818824
2025-01-01         110.571820
2025-02-01         110.571820
2025-03-01         110.571820
2025-04-01         118.732611
2025-05-01         118.732611
2025-06-01         118.732611
2025-07-01         121.352007
2025-08-01         121.352007
2025-09-01         121.352007
2025-10-01         117.830261
2025-11-01         117.830261
2025-12-01         117.830261
2026-01-01         119.639605
2026-02-01         119.639605
2026-03-01         119.639605


In [85]:
monthly_hpi["hpi_growth_12m_pct"] = (
    monthly_hpi["house_price_index"]
    .pct_change(periods=12) * 100
)

In [86]:
print(
    monthly_hpi[
        monthly_hpi["date"].between(
            "2023-01-01",
            "2025-03-01"
        )
    ].to_string(index=False)
)

      date  house_price_index  hpi_growth_12m_pct
2023-01-01         311.884308          209.499009
2023-01-01          96.697261          -67.985080
2023-02-01          96.697261          -67.985080
2023-02-01         311.884308          209.499009
2023-03-01         311.884308            3.259918
2023-03-01          96.697261           -4.042282
2023-04-01         102.247108           -0.497304
2023-04-01         308.595942            1.540405
2023-05-01         102.247108          -66.356613
2023-05-01         308.595942          200.312926
2023-06-01         102.247108          -66.356613
2023-06-01         308.595942          200.312926
2023-07-01         110.307286          -64.631986
2023-07-01         313.547776          224.257143
2023-08-01         110.307286           14.074882
2023-08-01         313.547776            0.533360
2023-09-01         110.307286          -64.631986
2023-09-01         313.547776          224.257143
2023-10-01         316.325244          209.373292


In [87]:
micro_clean = micro_clean.drop(
    columns=[
        "house_price_index",
        "hpi_growth_12m_pct"
    ],
    errors="ignore"
)

In [88]:
micro_clean = micro_clean.merge(
    monthly_hpi,
    on="date",
    how="left"
)

In [89]:
micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

In [90]:
print(
    micro_clean[
        micro_clean["date"].between(
            "2023-07-01",
            "2025-03-01"
        )
    ][
        [
            "date",
            "house_price_index",
            "hpi_growth_12m_pct"
        ]
    ].to_string(index=False)
)

      date  house_price_index  hpi_growth_12m_pct
2023-07-01         110.307286          -64.631986
2023-07-01         313.547776          224.257143
2023-08-01         110.307286           14.074882
2023-08-01         313.547776            0.533360
2023-09-01         110.307286          -64.631986
2023-09-01         313.547776          224.257143
2023-10-01         316.325244          209.373292
2023-10-01         114.402271          -62.928135
2023-11-01         114.402271           11.888027
2023-11-01         316.325244            2.504668
2023-12-01         114.402271          -62.928135
2023-12-01         316.325244          209.373292
2024-01-01         322.176912          192.072196
2024-01-01         114.369276          -63.524131
2024-02-01         322.176912          192.072196
2024-02-01         114.369276          -63.524131
2024-03-01         322.176912          192.072196
2024-03-01         114.369276          -63.524131
2024-04-01         109.967298          -65.236003


In [91]:
hpi_check = micro_clean[
    ["date", "house_price_index"]
].copy()

hpi_check["pct_change"] = (
    hpi_check["house_price_index"]
    .pct_change() * 100
)

print(
    hpi_check[
        hpi_check["pct_change"].abs() > 20
    ].to_string(index=False)
)

      date  house_price_index  pct_change
2022-01-01          98.969405  -65.942539
2022-01-01         296.609114  199.697784
2022-02-01          98.969405  -66.633053
2022-03-01         296.609114  199.697784
2022-03-01          98.969405  -66.633053
2022-04-01         298.029538  205.665773
2022-05-01          97.501770  -67.284528
2022-05-01         298.029538  205.665773
2022-06-01          97.501770  -67.284528
2022-07-01         302.038114  199.728119
2022-08-01         100.770697  -66.636430
2022-09-01         302.038114  199.728119
2022-10-01         102.758128  -65.978424
2022-10-01         303.914430  195.757071
2022-11-01         102.758128  -66.188467
2022-12-01         303.914430  195.757071
2022-12-01         102.758128  -66.188467
2023-01-01         311.884308  203.513030
2023-01-01          96.697261  -68.995792
2023-02-01         311.884308  222.536859
2023-03-01          96.697261  -68.995792
2023-04-01         308.595942  201.813858
2023-05-01         102.247108  -66

In [92]:
import pandas as pd

macro_path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\micro_indicators_monthly.csv"

micro_clean = pd.read_csv(
    macro_path,
    parse_dates=["date"]
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .drop_duplicates(subset=["date"])
    .reset_index(drop=True)
)

print(micro_clean.shape)
print(micro_clean.head())
print(micro_clean.tail())

(201, 12)
        date  oil_price_usd_bbl  gold_price_inr_10g  cpi_index  \
0 2010-01-01          71.459999        16050.981650        NaN   
1 2010-02-01          77.589996        16563.372599        NaN   
2 2010-03-01          82.699997        16035.455348        NaN   
3 2010-04-01          87.440002        16788.934644        NaN   
4 2010-05-01          74.019997        18067.165857        NaN   

   cpi_inflation_yoy_pct  house_price_index  oil_return_1m_pct  \
0                    NaN                NaN                NaN   
1                    NaN                NaN           8.578222   
2                    NaN                NaN           6.585901   
3                    NaN          90.626706           5.731567   
4                    NaN          90.626706         -15.347673   

   oil_return_12m_pct  gold_return_1m_pct  gold_return_12m_pct  \
0                 NaN                 NaN                  NaN   
1                 NaN            3.192272                  NaN  

In [93]:
print(hpi_quarterly.to_string(index=False))

   quarter  house_price_index
Q1.2010-11          94.239884
Q2.2010-11          99.811033
Q3.2010-11          99.404514
Q4.2010-11         106.632290
Q1.2011-12         115.987161
Q2.2011-12         119.435481
Q3.2011-12         125.476596
Q4.2011-12         134.100969
Q1.2012-13         142.625684
Q2.2012-13         147.095652
Q3.2012-13         156.973393
Q4.2012-13         160.840570
Q1.2013-14         162.277914
Q2.2013-14         169.235124
Q3.2013-14         172.810613
Q4.2013-14         180.492826
Q1.2014-15         188.044616
Q2.2014-15         193.049027
Q3.2014-15         202.036803
Q4.2014-15         212.098670
Q1.2015-16         215.309350
Q2.2015-16         218.185050
Q3.2015-16         221.710320
Q4.2015-16         219.137554
Q1.2016-17         231.085451
Q2.2016-17         234.892419
Q3.2016-17         240.164989
Q4.2016-17         242.002821
Q1.2017-18         251.198215
Q2.2017-18         252.358109
Q3.2017-18         257.510687
Q4.2017-18         258.228019
Q1.2018-19

In [94]:
hpi_rows = []

for row in rows[93:110]:

    quarter = row[1]

    if quarter is None:
        continue

    quarter = str(quarter).strip()

    # Keep only Q1/Q2/Q3/Q4 rows
    if not quarter.startswith("Q"):
        continue

    # All India is NOT in the new 18-city table.
    # The values we extracted earlier were from the
    # All India column in the corresponding section.
    all_india = row[12]

    if all_india is None:
        continue

    hpi_rows.append({
        "quarter": quarter,
        "house_price_index": float(all_india)
    })

hpi_quarterly = pd.DataFrame(hpi_rows)

print(hpi_quarterly.to_string(index=False))

        quarter  house_price_index
     Q1.2022-23          98.969405
     Q2.2022-23          97.501770
     Q3.2022-23         100.770697
     Q4.2022-23         102.758128
     Q1.2023-24          96.697261
     Q2.2023-24         102.247108
     Q3.2023-24         110.307286
     Q4.2023-24         114.402271
     Q1.2024-25         114.369276
     Q2.2024-25         109.967298
     Q3.2024-25         113.104928
     Q4.2024-25         105.818824
     Q1.2025-26         110.571820
     Q2.2025-26         118.732611
     Q3.2025-26         121.352007
     Q4.2025-26         117.830261
Q1.2026-27(P)**         119.639605


In [95]:
quarter_months = {
    "Q1": [1, 2, 3],
    "Q2": [4, 5, 6],
    "Q3": [7, 8, 9],
    "Q4": [10, 11, 12]
}

monthly_hpi = []

for _, row in hpi_quarterly.iterrows():

    quarter, financial_year = row["quarter"].split(".")

    start_year = int(financial_year.split("-")[0])

    for month in quarter_months[quarter]:

        monthly_hpi.append({
            "date": pd.Timestamp(start_year, month, 1),
            "house_price_index": row["house_price_index"]
        })

monthly_hpi = pd.DataFrame(monthly_hpi)

monthly_hpi["hpi_growth_12m_pct"] = (
    monthly_hpi["house_price_index"]
    .pct_change(periods=12) * 100
)

monthly_hpi = (
    monthly_hpi
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

print(monthly_hpi.tail(20).to_string(index=False))

      date  house_price_index  hpi_growth_12m_pct
2024-08-01         113.104928            2.536226
2024-09-01         113.104928            2.536226
2024-10-01         105.818824           -7.502865
2024-11-01         105.818824           -7.502865
2024-12-01         105.818824           -7.502865
2025-01-01         110.571820           -3.320346
2025-02-01         110.571820           -3.320346
2025-03-01         110.571820           -3.320346
2025-04-01         118.732611            7.970836
2025-05-01         118.732611            7.970836
2025-06-01         118.732611            7.970836
2025-07-01         121.352007            7.291530
2025-08-01         121.352007            7.291530
2025-09-01         121.352007            7.291530
2025-10-01         117.830261           11.350945
2025-11-01         117.830261           11.350945
2025-12-01         117.830261           11.350945
2026-01-01         119.639605            8.200810
2026-02-01         119.639605            8.200810


In [96]:
monthly_hpi = monthly_hpi[
    monthly_hpi["date"] >= "2022-04-01"
].copy()

In [97]:
micro_clean = micro_clean.drop(
    columns=[
        "house_price_index",
        "hpi_growth_12m_pct"
    ],
    errors="ignore"
)

micro_clean = micro_clean.merge(
    monthly_hpi,
    on="date",
    how="left",
    validate="one_to_one"
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

In [98]:
print("Shape:", micro_clean.shape)

print(
    micro_clean[
        micro_clean["date"].between(
            "2022-01-01",
            "2026-09-01"
        )
    ][
        ["date", "house_price_index", "hpi_growth_12m_pct"]
    ].to_string(index=False)
)

Shape: (201, 12)
      date  house_price_index  hpi_growth_12m_pct
2022-01-01                NaN                 NaN
2022-02-01                NaN                 NaN
2022-03-01                NaN                 NaN
2022-04-01          97.501770                 NaN
2022-05-01          97.501770                 NaN
2022-06-01          97.501770                 NaN
2022-07-01         100.770697                 NaN
2022-08-01         100.770697                 NaN
2022-09-01         100.770697                 NaN
2022-10-01         102.758128                 NaN
2022-11-01         102.758128                 NaN
2022-12-01         102.758128                 NaN
2023-01-01          96.697261           -2.295804
2023-02-01          96.697261           -2.295804
2023-03-01          96.697261           -2.295804
2023-04-01         102.247108            4.866925
2023-05-01         102.247108            4.866925
2023-06-01         102.247108            4.866925
2023-07-01         110.307286    

In [99]:
hpi_rows = []

for row in rows[93:110]:

    quarter = row[1]

    if quarter is None:
        continue

    quarter = str(quarter).strip()

    # Remove provisional marker
    quarter = quarter.replace("(P)**", "")

    if not quarter.startswith("Q"):
        continue

    # All India value
    all_india = row[12]

    if all_india is None:
        continue

    hpi_rows.append({
        "quarter": quarter,
        "house_price_index": float(all_india)
    })

hpi_quarterly = pd.DataFrame(hpi_rows)

print(hpi_quarterly.to_string(index=False))

   quarter  house_price_index
Q1.2022-23          98.969405
Q2.2022-23          97.501770
Q3.2022-23         100.770697
Q4.2022-23         102.758128
Q1.2023-24          96.697261
Q2.2023-24         102.247108
Q3.2023-24         110.307286
Q4.2023-24         114.402271
Q1.2024-25         114.369276
Q2.2024-25         109.967298
Q3.2024-25         113.104928
Q4.2024-25         105.818824
Q1.2025-26         110.571820
Q2.2025-26         118.732611
Q3.2025-26         121.352007
Q4.2025-26         117.830261
Q1.2026-27         119.639605


In [100]:
quarter_months = {
    "Q1": [1, 2, 3],
    "Q2": [4, 5, 6],
    "Q3": [7, 8, 9],
    "Q4": [10, 11, 12]
}

monthly_hpi = []

for _, row in hpi_quarterly.iterrows():

    quarter, financial_year = row["quarter"].split(".")

    start_year = int(financial_year.split("-")[0])

    for month in quarter_months[quarter]:

        monthly_hpi.append({
            "date": pd.Timestamp(start_year, month, 1),
            "house_price_index": row["house_price_index"]
        })

monthly_hpi = pd.DataFrame(monthly_hpi)

monthly_hpi["hpi_growth_12m_pct"] = (
    monthly_hpi["house_price_index"]
    .pct_change(periods=12) * 100
)

monthly_hpi = (
    monthly_hpi
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

In [101]:
micro_clean = pd.read_csv(
    macro_path,
    parse_dates=["date"]
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .drop_duplicates(subset=["date"])
    .reset_index(drop=True)
)

micro_clean = micro_clean.drop(
    columns=["house_price_index", "hpi_growth_12m_pct"],
    errors="ignore"
)

micro_clean = micro_clean.merge(
    monthly_hpi,
    on="date",
    how="left",
    validate="one_to_one"
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

In [102]:
print(
    micro_clean[
        micro_clean["date"].between(
            "2022-04-01",
            "2024-06-01"
        )
    ][
        ["date", "house_price_index", "hpi_growth_12m_pct"]
    ].to_string(index=False)
)

      date  house_price_index  hpi_growth_12m_pct
2022-04-01          97.501770                 NaN
2022-05-01          97.501770                 NaN
2022-06-01          97.501770                 NaN
2022-07-01         100.770697                 NaN
2022-08-01         100.770697                 NaN
2022-09-01         100.770697                 NaN
2022-10-01         102.758128                 NaN
2022-11-01         102.758128                 NaN
2022-12-01         102.758128                 NaN
2023-01-01          96.697261           -2.295804
2023-02-01          96.697261           -2.295804
2023-03-01          96.697261           -2.295804
2023-04-01         102.247108            4.866925
2023-05-01         102.247108            4.866925
2023-06-01         102.247108            4.866925
2023-07-01         110.307286            9.463653
2023-08-01         110.307286            9.463653
2023-09-01         110.307286            9.463653
2023-10-01         114.402271           11.331603


In [103]:
# Recreate the new-base All India HPI explicitly

hpi_rows = []

for row in rows:

    quarter = row[1]

    if quarter is None:
        continue

    quarter = str(quarter).strip()

    # Remove provisional marker
    clean_quarter = quarter.replace("(P)**", "").strip()

    # Only target the new-base financial years
    valid_years = [
        "2022-23",
        "2023-24",
        "2024-25",
        "2025-26",
        "2026-27"
    ]

    if not any(clean_quarter.endswith(year) for year in valid_years):
        continue

    if not clean_quarter.startswith(("Q1.", "Q2.", "Q3.", "Q4.")):
        continue

    # Column 12 = All India in the relevant section
    all_india = row[12]

    if all_india is None:
        continue

    hpi_rows.append({
        "quarter": clean_quarter,
        "house_price_index": float(all_india)
    })

hpi_quarterly = pd.DataFrame(hpi_rows)

print(hpi_quarterly.to_string(index=False))

   quarter  house_price_index
Q1.2022-23         296.609114
Q2.2022-23         298.029538
Q3.2022-23         302.038114
Q4.2022-23         303.914430
Q1.2023-24         311.884308
Q2.2023-24         308.595942
Q3.2023-24         313.547776
Q4.2023-24         316.325244
Q1.2024-25         322.176912
Q2.2024-25         321.999921
Q3.2024-25         323.255379
Q4.2024-25         326.229666
Q1.2022-23          98.969405
Q2.2022-23          97.501770
Q3.2022-23         100.770697
Q4.2022-23         102.758128
Q1.2023-24          96.697261
Q2.2023-24         102.247108
Q3.2023-24         110.307286
Q4.2023-24         114.402271
Q1.2024-25         114.369276
Q2.2024-25         109.967298
Q3.2024-25         113.104928
Q4.2024-25         105.818824
Q1.2025-26         110.571820
Q2.2025-26         118.732611
Q3.2025-26         121.352007
Q4.2025-26         117.830261
Q1.2026-27         119.639605


In [105]:
print(hpi_quarterly["quarter"].tolist())

['Q1.2022-23', 'Q2.2022-23', 'Q3.2022-23', 'Q4.2022-23', 'Q1.2023-24', 'Q2.2023-24', 'Q3.2023-24', 'Q4.2023-24', 'Q1.2024-25', 'Q2.2024-25', 'Q3.2024-25', 'Q4.2024-25', 'Q1.2022-23', 'Q2.2022-23', 'Q3.2022-23', 'Q4.2022-23', 'Q1.2023-24', 'Q2.2023-24', 'Q3.2023-24', 'Q4.2023-24', 'Q1.2024-25', 'Q2.2024-25', 'Q3.2024-25', 'Q4.2024-25', 'Q1.2025-26', 'Q2.2025-26', 'Q3.2025-26', 'Q4.2025-26', 'Q1.2026-27']


In [3]:
# ============================================================
# BLOCK 1
# COMPLETE MACRO DATA CLEANING
# ============================================================

import pandas as pd
import numpy as np
from openpyxl import load_workbook


# ============================================================
# 1. FILE PATHS
# ============================================================

MACRO_PATH = (
    r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence"
    r"\data\processed\final\micro_indicators_monthly.csv"
)

CPI_PATH = (
    r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence"
    r"\data\raw\cpi_887.xlsx"
)

HPI_PATH = (
    r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence"
    r"\data\raw\House Price Index Publication(1).xlsx"
)

OUTPUT_PATH = (
    r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence"
    r"\data\processed\final\micro_indicators_monthly_clean.csv"
)


# ============================================================
# 2. LOAD ORIGINAL MACRO DATA
# ============================================================

print("=" * 70)
print("LOADING MACRO DATA")
print("=" * 70)

micro_clean = pd.read_csv(MACRO_PATH)

micro_clean["date"] = pd.to_datetime(
    micro_clean["date"],
    errors="coerce"
)

micro_clean = (
    micro_clean
    .sort_values("date")
    .drop_duplicates(
        subset=["date"]
    )
    .reset_index(drop=True)
)

print("Shape:", micro_clean.shape)

print(
    "Date range:",
    micro_clean["date"].min(),
    "→",
    micro_clean["date"].max()
)


# ============================================================
# 3. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "oil_price_usd_bbl",
    "gold_price_inr_10g",
    "cpi_index",
    "cpi_inflation_yoy_pct",
    "house_price_index",
    "oil_return_1m_pct",
    "oil_return_12m_pct",
    "gold_return_1m_pct",
    "gold_return_12m_pct",
    "inflation_change_1m",
    "hpi_growth_12m_pct",
]

for col in numeric_columns:

    if col in micro_clean.columns:

        micro_clean[col] = pd.to_numeric(
            micro_clean[col],
            errors="coerce"
        )


# ============================================================
# 4. OIL CLEANING
# ============================================================

print("\n" + "=" * 70)
print("OIL CLEANING")
print("=" * 70)

print(
    "Missing oil before:",
    micro_clean["oil_price_usd_bbl"].isna().sum()
)


# Linear interpolation for isolated missing prices
micro_clean["oil_price_usd_bbl"] = (
    micro_clean["oil_price_usd_bbl"]
    .interpolate(method="linear")
)


# Recalculate returns
micro_clean["oil_return_1m_pct"] = (
    micro_clean["oil_price_usd_bbl"]
    .pct_change(1)
    * 100
)

micro_clean["oil_return_12m_pct"] = (
    micro_clean["oil_price_usd_bbl"]
    .pct_change(12)
    * 100
)


print(
    "Missing oil after:",
    micro_clean["oil_price_usd_bbl"].isna().sum()
)


# ============================================================
# 5. GOLD CLEANING
# ============================================================

print("\n" + "=" * 70)
print("GOLD CLEANING")
print("=" * 70)

print(
    "Missing gold before:",
    micro_clean["gold_price_inr_10g"].isna().sum()
)


micro_clean["gold_price_inr_10g"] = (
    micro_clean["gold_price_inr_10g"]
    .interpolate(method="linear")
)


# Recalculate returns
micro_clean["gold_return_1m_pct"] = (
    micro_clean["gold_price_inr_10g"]
    .pct_change(1)
    * 100
)

micro_clean["gold_return_12m_pct"] = (
    micro_clean["gold_price_inr_10g"]
    .pct_change(12)
    * 100
)


print(
    "Missing gold after:",
    micro_clean["gold_price_inr_10g"].isna().sum()
)


# ============================================================
# 6. CPI CLEANING
# ============================================================

print("\n" + "=" * 70)
print("CPI CLEANING")
print("=" * 70)


micro_clean["cpi_index"] = pd.to_numeric(
    micro_clean["cpi_index"],
    errors="coerce"
)

micro_clean["cpi_inflation_yoy_pct"] = pd.to_numeric(
    micro_clean["cpi_inflation_yoy_pct"],
    errors="coerce"
)


# ------------------------------------------------------------
# Calculate CPI inflation from CPI index
# ------------------------------------------------------------

calculated_cpi_inflation = (
    micro_clean["cpi_index"]
    .pct_change(12)
    * 100
)


# Fill only where an existing CPI index allows calculation
micro_clean["cpi_inflation_yoy_pct"] = (
    micro_clean["cpi_inflation_yoy_pct"]
    .fillna(calculated_cpi_inflation)
)


# ============================================================
# 7. LOAD NEW CPI 2024=100 DATA
# ============================================================

print("\nLoading new CPI workbook...")

wb_cpi = load_workbook(
    CPI_PATH,
    read_only=True,
    data_only=True
)

ws_cpi = wb_cpi["CPI Data"]

cpi_rows = list(ws_cpi.values)

cpi_headers = cpi_rows[0]

cpi_new = pd.DataFrame(
    cpi_rows[1:],
    columns=cpi_headers
)


print(
    "New CPI shape:",
    cpi_new.shape
)


# ============================================================
# 8. CLEAN CPI FIELDS
# ============================================================

for col in [
    "state",
    "sector",
    "year",
    "month"
]:

    cpi_new[col] = (
        cpi_new[col]
        .astype(str)
        .str.strip()
    )


# ============================================================
# 9. GET ALL INDIA + COMBINED CPI
# ============================================================

cpi_new = cpi_new[
    (cpi_new["state"] == "All India") &
    (cpi_new["sector"] == "Combined")
].copy()


# ============================================================
# 10. CREATE CPI DATE
# ============================================================

cpi_new["date"] = pd.to_datetime(
    cpi_new["year"]
    + "-"
    + cpi_new["month"],
    format="%Y-%B",
    errors="coerce"
)


cpi_new["inflation"] = pd.to_numeric(
    cpi_new["inflation"],
    errors="coerce"
)


cpi_new = (
    cpi_new[
        [
            "date",
            "inflation"
        ]
    ]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)


# ============================================================
# 11. GET 2026 CPI
# ============================================================

cpi_2026 = cpi_new[
    (
        cpi_new["date"].dt.year == 2026
    )
    &
    (
        cpi_new["date"].dt.month <= 8
    )
].copy()


cpi_2026 = cpi_2026.rename(
    columns={
        "inflation":
        "cpi_inflation_new"
    }
)


print("\n2026 CPI:")
print(
    cpi_2026.to_string(
        index=False
    )
)


# ============================================================
# 12. MERGE NEW CPI
# ============================================================

micro_clean = micro_clean.merge(
    cpi_2026[
        [
            "date",
            "cpi_inflation_new"
        ]
    ],
    on="date",
    how="left"
)


micro_clean["cpi_inflation_yoy_pct"] = (
    micro_clean[
        "cpi_inflation_yoy_pct"
    ]
    .fillna(
        micro_clean[
            "cpi_inflation_new"
        ]
    )
)


micro_clean = micro_clean.drop(
    columns=[
        "cpi_inflation_new"
    ]
)


# ============================================================
# 13. RECALCULATE INFLATION CHANGE
# ============================================================

micro_clean["cpi_inflation_yoy_pct"] = (
    pd.to_numeric(
        micro_clean[
            "cpi_inflation_yoy_pct"
        ],
        errors="coerce"
    )
)


micro_clean["inflation_change_1m"] = (
    micro_clean[
        "cpi_inflation_yoy_pct"
    ].diff()
)


# ============================================================
# 14. LOAD HPI WORKBOOK
# ============================================================

print("\n" + "=" * 70)
print("LOADING HPI DATA")
print("=" * 70)


wb_hpi = load_workbook(
    HPI_PATH,
    read_only=True,
    data_only=True
)

ws_hpi = wb_hpi["House Price Index"]

hpi_rows = list(ws_hpi.values)

print(
    "Total Excel rows:",
    len(hpi_rows)
)


# ============================================================
# 15. DEFINE 18 CITIES
# ============================================================

cities = [
    "Ahmedabad",
    "Bangalore",
    "Chandigarh",
    "Chennai",
    "Delhi",
    "Gautam Buddha Nagar",
    "Ghaziabad",
    "Hyderabad",
    "Jaipur",
    "Kanpur",
    "Kochi",
    "Kolkata",
    "Lucknow",
    "Mumbai",
    "Nagpur",
    "Pune",
    "Thane",
    "Thiruvananthapuram",
]


# ============================================================
# 16. FIND 2022-23 BASE YEAR
# ============================================================

base_row = None

for i, row in enumerate(hpi_rows):

    row_text = " ".join(
        str(x).strip()
        for x in row
        if x is not None
    )

    if (
        "Base Year" in row_text
        and "2022-23" in row_text
    ):

        base_row = i
        break


if base_row is None:

    raise RuntimeError(
        "Could not find 2022-23 HPI base section."
    )


print(
    "New HPI base section:",
    base_row + 1
)


# ============================================================
# 17. FIND 18-CITY HEADER
# ============================================================

header_row = None

for i in range(
    base_row,
    min(
        base_row + 10,
        len(hpi_rows)
    )
):

    row = hpi_rows[i]

    row_values = [
        str(x).strip()
        for x in row
        if x is not None
    ]

    matches = sum(
        city in row_values
        for city in cities
    )

    if matches >= 15:

        header_row = i
        break


if header_row is None:

    raise RuntimeError(
        "Could not find HPI city header."
    )


print(
    "HPI city header:",
    header_row + 1
)


# ============================================================
# 18. FIND CITY COLUMNS
# ============================================================

header = hpi_rows[header_row]

city_columns = {}


for idx, value in enumerate(header):

    if value is None:
        continue

    value = str(value).strip()

    if value in cities:

        city_columns[value] = idx


print("\nCities found:")

for city in cities:

    print(
        f"{city}: column {city_columns.get(city)}"
    )


if len(city_columns) != 18:

    raise RuntimeError(
        f"Expected 18 cities, found {len(city_columns)}"
    )


# ============================================================
# 19. EXTRACT QUARTERLY HPI
# ============================================================

quarterly_records = []


for row in hpi_rows[header_row + 1:]:

    if row is None:
        continue

    if len(row) < 20:
        continue


    # ========================================================
    # IMPORTANT:
    # QUARTER IS COLUMN 1
    # CITIES ARE COLUMNS 2-19
    # ========================================================

    quarter = row[1]


    if quarter is None:
        continue


    quarter = str(
        quarter
    ).strip()


    # Check Q1/Q2/Q3/Q4
    if not (
        quarter.startswith("Q1.")
        or quarter.startswith("Q2.")
        or quarter.startswith("Q3.")
        or quarter.startswith("Q4.")
    ):

        continue


    # Validate quarter
    try:

        q_number = int(
            quarter[1]
        )

        fy_start = int(
            quarter
            .split(".")[1]
            .split("-")[0]
        )

    except Exception:

        continue


    if q_number not in [1, 2, 3, 4]:
        continue

    if fy_start < 2022:
        continue


    # ========================================================
    # EXTRACT CITY VALUES
    # ========================================================

    record = {
        "quarter": quarter
    }

    valid_count = 0


    for city in cities:

        col = city_columns[city]

        value = row[col]

        value = pd.to_numeric(
            value,
            errors="coerce"
        )

        record[city] = value

        if pd.notna(value):
            valid_count += 1


    # Need enough valid city observations
    if valid_count >= 15:

        quarterly_records.append(
            record
        )


# ============================================================
# 20. CREATE QUARTERLY DATAFRAME
# ============================================================

hpi_quarterly = pd.DataFrame(
    quarterly_records,
    columns=[
        "quarter"
    ] + cities
)


# Remove duplicates
hpi_quarterly = (
    hpi_quarterly
    .drop_duplicates(
        subset=["quarter"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 21. SORT QUARTERS
# ============================================================

def quarter_sort_key(q):

    q_number = int(q[1])

    year = int(
        q.split(".")[1]
        .split("-")[0]
    )

    return (
        year,
        q_number
    )


if not hpi_quarterly.empty:

    hpi_quarterly["_sort"] = (
        hpi_quarterly["quarter"]
        .apply(
            quarter_sort_key
        )
    )

    hpi_quarterly = (
        hpi_quarterly
        .sort_values("_sort")
        .drop(columns="_sort")
        .reset_index(drop=True)
    )


# ============================================================
# 22. VALIDATE QUARTERLY HPI
# ============================================================

print("\n" + "=" * 70)
print("QUARTERLY HPI")
print("=" * 70)

print(
    "Shape:",
    hpi_quarterly.shape
)

print(
    "\nQuarters:"
)

print(
    hpi_quarterly[
        "quarter"
    ].tolist()
)


if hpi_quarterly.empty:

    raise RuntimeError(
        """
HPI extraction returned ZERO quarters.

The workbook structure is not being
read as expected. Stop here and inspect
hpi_rows around the city header.
"""
    )


# ============================================================
# 23. CREATE 18-CITY EQUAL-WEIGHTED COMPOSITE
# ============================================================

hpi_quarterly[
    "house_price_index"
] = (
    hpi_quarterly[
        cities
    ]
    .mean(
        axis=1,
        skipna=True
    )
)


hpi_composite = hpi_quarterly[
    [
        "quarter",
        "house_price_index"
    ]
].copy()


print("\n" + "=" * 70)
print("HPI COMPOSITE")
print("=" * 70)

print(
    hpi_composite.to_string(
        index=False
    )
)


# ============================================================
# 24. CONVERT FINANCIAL-YEAR QUARTER TO CALENDAR MONTH
# ============================================================

def quarter_to_start_date(quarter):

    q = int(
        quarter[1]
    )

    fy = int(
        quarter
        .split(".")[1]
        .split("-")[0]
    )


    if q == 1:

        # Apr-Jun
        return pd.Timestamp(
            year=fy,
            month=4,
            day=1
        )


    elif q == 2:

        # Jul-Sep
        return pd.Timestamp(
            year=fy,
            month=7,
            day=1
        )


    elif q == 3:

        # Oct-Dec
        return pd.Timestamp(
            year=fy,
            month=10,
            day=1
        )


    elif q == 4:

        # Jan-Mar
        return pd.Timestamp(
            year=fy + 1,
            month=1,
            day=1
        )


    raise ValueError(
        f"Invalid quarter: {quarter}"
    )


# ============================================================
# 25. CREATE MONTHLY HPI
# ============================================================

monthly_records = []


for _, row in hpi_composite.iterrows():

    start_date = (
        quarter_to_start_date(
            row["quarter"]
        )
    )

    value = row[
        "house_price_index"
    ]


    for month_offset in range(3):

        current_date = (
            start_date
            + pd.DateOffset(
                months=month_offset
            )
        )


        monthly_records.append(
            {
                "date": current_date,
                "house_price_index": value
            }
        )


# Explicit columns
hpi_monthly = pd.DataFrame(
    monthly_records,
    columns=[
        "date",
        "house_price_index"
    ]
)


# ============================================================
# 26. CLEAN MONTHLY HPI
# ============================================================

hpi_monthly["date"] = pd.to_datetime(
    hpi_monthly["date"],
    errors="coerce"
)


hpi_monthly[
    "house_price_index"
] = pd.to_numeric(
    hpi_monthly[
        "house_price_index"
    ],
    errors="coerce"
)


hpi_monthly = (
    hpi_monthly
    .dropna(
        subset=["date"]
    )
    .drop_duplicates(
        subset=["date"]
    )
    .sort_values(
        "date"
    )
    .reset_index(drop=True)
)


# ============================================================
# 27. CALCULATE HPI 12-MONTH GROWTH
# ============================================================

hpi_monthly[
    "hpi_growth_12m_pct"
] = (
    hpi_monthly[
        "house_price_index"
    ]
    .pct_change(
        periods=12
    )
    * 100
)


# ============================================================
# 28. CHECK MONTHLY HPI
# ============================================================

print("\n" + "=" * 70)
print("MONTHLY HPI")
print("=" * 70)

print(
    "Shape:",
    hpi_monthly.shape
)

print(
    "First date:",
    hpi_monthly["date"].min()
)

print(
    "Last date:",
    hpi_monthly["date"].max()
)

print(
    "\nFirst observations:"
)

print(
    hpi_monthly
    .head(12)
    .to_string(index=False)
)

print(
    "\nLast observations:"
)

print(
    hpi_monthly
    .tail(12)
    .to_string(index=False)
)


# ============================================================
# 29. REMOVE OLD HPI FROM MACRO
# ============================================================

micro_clean = micro_clean.drop(
    columns=[
        "house_price_index",
        "hpi_growth_12m_pct"
    ],
    errors="ignore"
)


# ============================================================
# 30. MERGE HPI
# ============================================================

micro_clean = micro_clean.merge(
    hpi_monthly[
        [
            "date",
            "house_price_index",
            "hpi_growth_12m_pct"
        ]
    ],
    on="date",
    how="left"
)


micro_clean = (
    micro_clean
    .sort_values("date")
    .reset_index(drop=True)
)

LOADING MACRO DATA
Shape: (201, 12)
Date range: 2010-01-01 00:00:00 → 2026-09-01 00:00:00

OIL CLEANING
Missing oil before: 29
Missing oil after: 0

GOLD CLEANING
Missing gold before: 29
Missing gold after: 0

CPI CLEANING

Loading new CPI workbook...
New CPI shape: (2200, 15)

2026 CPI:
      date  cpi_inflation_new
2026-01-01               2.74
2026-02-01               3.21
2026-03-01               3.40
2026-04-01               3.48
2026-05-01               3.93
2026-06-01               4.38
2026-07-01               4.45
2026-08-01               4.82

LOADING HPI DATA
Total Excel rows: 112
New HPI base section: 91
HPI city header: 93

Cities found:
Ahmedabad: column 2
Bangalore: column 3
Chandigarh: column 4
Chennai: column 5
Delhi: column 6
Gautam Buddha Nagar: column 7
Ghaziabad: column 8
Hyderabad: column 9
Jaipur: column 10
Kanpur: column 11
Kochi: column 12
Kolkata: column 13
Lucknow: column 14
Mumbai: column 15
Nagpur: column 16
Pune: column 17
Thane: column 18
Thiruvananthapur

c:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\backend\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
# ==========================================
# SAVE UPDATED MACRO DATA
# ==========================================

output_path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\micro_indicators_monthly_clean.parquet"

micro_clean.to_parquet(output_path, index=False)

print("\n" + "=" * 60)
print("UPDATED MACRO DATA SAVED")
print("=" * 60)

print("File:", output_path)
print("Shape:", micro_clean.shape)
print("Date range:",
      micro_clean["date"].min().date(),
      "→",
      micro_clean["date"].max().date())

print("\nMissing values:")
print(micro_clean.isna().sum())


UPDATED MACRO DATA SAVED
File: C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\micro_indicators_monthly_clean.parquet
Shape: (201, 12)
Date range: 2010-01-01 → 2026-09-01

Missing values:
date                       0
oil_price_usd_bbl          0
gold_price_inr_10g         0
cpi_index                 45
cpi_inflation_yoy_pct     49
oil_return_1m_pct          1
oil_return_12m_pct        12
gold_return_1m_pct         1
gold_return_12m_pct       12
inflation_change_1m       50
house_price_index        150
hpi_growth_12m_pct       162
dtype: int64


In [ ]:
import pandas as pd

path = r"C:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed\final\micro_indicators_monthly_clean.parquet"

micro_clean = pd.read_parquet(path)

print("=" * 60)
print("UPDATED MICRO DATASET")
print("=" * 60)

print("Shape:", micro_clean.shape)
print("Date range:",
      micro_clean["date"].min(),
      "→",
      micro_clean["date"].max())

print("\nColumns:")
print(micro_clean.columns.tolist())

print("\nMissing values:")
print(micro_clean.isna().sum())

print("\nLast 10 rows:")
display(micro_clean.tail(10))

UPDATED MICRO DATASET
Shape: (201, 12)
Date range: 2010-01-01 00:00:00 → 2026-09-01 00:00:00

Columns:
['date', 'oil_price_usd_bbl', 'gold_price_inr_10g', 'cpi_index', 'cpi_inflation_yoy_pct', 'oil_return_1m_pct', 'oil_return_12m_pct', 'gold_return_1m_pct', 'gold_return_12m_pct', 'inflation_change_1m', 'house_price_index', 'hpi_growth_12m_pct']

Missing values:
date                       0
oil_price_usd_bbl          0
gold_price_inr_10g         0
cpi_index                 45
cpi_inflation_yoy_pct     49
oil_return_1m_pct          1
oil_return_12m_pct        12
gold_return_1m_pct         1
gold_return_12m_pct       12
inflation_change_1m       50
house_price_index        150
hpi_growth_12m_pct       162
dtype: int64

Last 10 rows:


,date,oil_price_usd_bbl,gold_price_inr_10g,cpi_index,cpi_inflation_yoy_pct,oil_return_1m_pct,oil_return_12m_pct,gold_return_1m_pct,gold_return_12m_pct,inflation_change_1m,house_price_index,hpi_growth_12m_pct
191,2025-12-01,60.849998,124843.449049,198.0,1.33,-3.718358,-18.704079,3.032286,65.809869,0.62,119.356591,6.952031
192,2026-01-01,70.690002,139869.146486,NaN,2.74,16.170919,-7.907764,12.035631,78.519140,1.41,119.520994,8.366470
193,2026-02-01,85.130002,140339.001746,NaN,3.21,20.427217,16.329601,0.335925,76.201677,0.47,119.520994,8.366470
194,2026-03-01,99.570002,140808.857005,NaN,3.40,16.962292,33.221842,0.334800,63.957605,0.19,119.520994,8.366470
195,2026-04-01,114.010002,141278.712265,NaN,3.48,14.502360,80.624214,0.333683,56.107396,0.08,121.860713,8.119030
196,2026-05-01,92.050003,140289.387658,NaN,3.93,-19.261467,44.053209,-0.700264,55.144986,0.45,121.860713,8.119030
197,2026-06-01,72.919998,123076.729941,NaN,4.38,-20.782188,6.897307,-12.269394,34.292945,0.45,121.860713,8.119030
198,2026-07-01,90.120003,126345.190545,NaN,4.45,23.587500,24.252040,2.655628,36.043459,0.07,NaN,NaN
199,2026-08-01,90.489998,137427.345530,NaN,4.82,0.410558,32.839099,8.771331,39.567317,0.37,NaN,NaN
200,2026-09-01,108.750000,133720.467561,NaN,NaN,20.179028,62.265004,-2.697337,22.078584,NaN,NaN,NaN


: 